# Capítulo 5 · Actividad 05: selección de características en enfermedad cardiaca

> ¿Necesito todas las variables del proyecto 1 para detectar enfermedad cardiaca?

Aplico la clase 09 (selección de características) a los datos de enfermedad cardiaca del proyecto 1 del capítulo 5. Sigo los mismos pasos de la clase: primero cada técnica de selección, luego comparo qué columnas eligió cada una y al final entreno SVC, regresión logística y Random Forest con cada subconjunto.

Del proyecto conservo la tabla `data/heart_disease_uci.csv`, la limpieza, el target y las 12 variables de entrada. `ca` y `dataset` siguen fuera, por las decisiones 6 y 7 del proyecto.

In [ ]:
from itertools import combinations
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn import set_config
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    RFE, SelectKBest, SequentialFeatureSelector,
    VarianceThreshold, f_classif,
)
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Lasso, LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, fbeta_score, roc_auc_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.svm import SVC

In [ ]:
set_config(transform_output="pandas")
sns.set_theme(style="whitegrid")
RANDOM_STATE = 42
N_COLUMNAS = 6

---
## Preparación

### Datos y limpieza del proyecto

In [ ]:
raiz = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").exists()), Path.cwd())
DATA_DIR = raiz / "data"
df = pd.read_csv(DATA_DIR / "heart_disease_uci.csv")
df.shape

Las mismas reglas del proyecto: un colesterol o una presión de 0 son faltantes mal capturados, `fbs` y `exang` pasan a texto y `target` vale 1 si `num` es 1, 2, 3 o 4.

In [ ]:
df["chol"] = df["chol"].replace(0, np.nan)
df["trestbps"] = df["trestbps"].replace(0, np.nan)
for col in ["fbs", "exang"]:
    df[col] = df[col].map({True: "sí", False: "no"})
df["target"] = (df["num"] > 0).astype(int)

In [ ]:
NUMERICAS = ["age", "trestbps", "chol", "thalch", "oldpeak"]
CATEGORICAS = ["sex", "cp", "fbs", "restecg", "exang", "slope", "thal"]

Faltantes por columna

In [ ]:
df[NUMERICAS + CATEGORICAS].isna().sum()

### Entrenamiento y prueba: 80-20

Hago la misma división que la clase: 80% para entrenamiento y 20% para prueba. `stratify=y` mantiene la proporción de enfermos y sanos, y `random_state=42` permite repetir la división. Como el proyecto también apartó primero un 20% con la misma semilla, la prueba son los mismos 184 pacientes del proyecto. La diferencia es que aquí no hay conjunto de validación: como en la clase, todo se decide con validación cruzada dentro de entrenamiento.

In [ ]:
X = df[NUMERICAS + CATEGORICAS]
y = df["target"]

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE,
)

In [ ]:
print("Entrenamiento:", X_train.shape)
print("Prueba:", X_test.shape)

In [ ]:
y_train.value_counts().sort_index()

### Faltantes y categorías

Las técnicas de la clase necesitan números sin faltantes. Uso el mismo preprocesamiento del proyecto: mediana en las numéricas, `faltante` como categoría propia y `OneHotEncoder` en las categóricas. Lo ajusto solo con entrenamiento y lo aplico a las dos partes.

In [ ]:
preprocesador = ColumnTransformer([
    ("num", SimpleImputer(strategy="median"), NUMERICAS),
    ("cat", Pipeline([
        ("imputador", SimpleImputer(strategy="constant", fill_value="faltante")),
        ("codificador", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
    ]), CATEGORICAS),
])
preprocesador.fit(X_train)

In [ ]:
X_train_prep = preprocesador.transform(X_train)
X_test_prep = preprocesador.transform(X_test)
X_train_prep.shape

In [ ]:
X_train_prep.columns.tolist()

**Qué veo.** Las 12 variables se vuelven 28 columnas. Las 5 numéricas siguen igual (`num__`) y cada categórica se abre en una columna de 0 y 1 por categoría (`cat__`), incluida `faltante` donde había huecos. Las técnicas de la clase van a elegir entre estas 28.

### Con qué métrica comparo

En el proyecto elegí con F2. Antes de seguir con esa métrica, reviso cuánto saca decirle "enfermo" a todos los pacientes de entrenamiento.

In [ ]:
todos_enfermos = np.ones(len(y_train))
print("F2 de decirle enfermo a todos:", round(fbeta_score(y_train, todos_enfermos, beta=2), 4))

**Qué veo.** Decirle enfermo a todos ya saca 0.8608 de F2, más que el 0.8541 que sacó mi regresión logística en validación cruzada en el proyecto. Pasa porque el 55.3% de entrenamiento está enfermo y F2 cuenta el recall el doble. Con F2 no se nota la diferencia entre un buen subconjunto y uno que solo hace que el modelo diga enfermo a casi todos.

Por eso aquí comparo con **AUC**, como en la actividad 01 del curso. El AUC mide qué tan seguido el modelo le da más puntaje a un enfermo que a un sano: 0.5 es lo mismo que ordenarlos al azar (es lo que pasa si a todos les doy el mismo puntaje, como al decirles enfermo a todos) y 1 es ordenarlos siempre bien.

---
## 1. Filtro de varianza

`VarianceThreshold(threshold=0)` quita solo las columnas que tienen siempre el mismo valor. No usa el target. Como en la clase, va antes del escalado.

In [ ]:
selector_varianza = VarianceThreshold(threshold=0)

In [ ]:
selector_varianza.fit(X_train_prep)

In [ ]:
columnas_varianza = X_train_prep.columns[selector_varianza.get_support()].tolist()
len(columnas_varianza)

**Qué veo.** Se quedan las 28. Ninguna columna es constante en entrenamiento.

---
## 2. Filtro de correlación entre características

La misma regla de la clase: recorro las columnas en su orden y conservo una si su correlación absoluta con todas las ya conservadas es menor que 0.85.

In [ ]:
correlacion = X_train_prep.corr().abs()

In [ ]:
fig, ax = plt.subplots(figsize=(12, 10))
sns.heatmap(correlacion, cmap="Reds", vmin=0, vmax=1, ax=ax)
ax.set_title("Correlación absoluta entre columnas de entrenamiento")
plt.tight_layout()
plt.show()

In [ ]:
columnas_correlacion = []

for columna in X_train_prep.columns:
    correlaciones = correlacion.loc[columna, columnas_correlacion]
    if (correlaciones < 0.85).all():
        columnas_correlacion.append(columna)

In [ ]:
print("Se quedan:", len(columnas_correlacion))
print("Se quitan:", [c for c in X_train_prep.columns if c not in columnas_correlacion])

In [ ]:
correlacion.loc[["cat__sex_Female", "cat__exang_no"], ["cat__sex_Male", "cat__exang_sí"]].round(3)

**Qué veo.** Se quitan dos columnas: `sex_Male` y `exang_sí`. `sex_Male` tiene correlación absoluta de 1 con `sex_Female` porque `sex` no tiene faltantes: si no es mujer, es hombre, así que las dos columnas dicen exactamente lo mismo. `exang_sí` y `exang_no` tienen 0.876; no llegan a 1 porque a algunos pacientes les falta `exang` y caen en `exang_faltante`.

Los demás cuadros oscuros fuera de la diagonal también son casi siempre categorías de una misma variable, como `fbs_no` y `fbs_sí`, pero quedan abajo de 0.85.

---
## 3. Selección univariada: ANOVA

`f_classif` revisa cada columna por separado: qué tanto cambia su promedio entre enfermos y sanos contra lo que varía dentro de cada grupo. `SelectKBest` conserva las 6 con mayor puntuación, como en la clase.

In [ ]:
selector_anova = SelectKBest(score_func=f_classif, k=N_COLUMNAS)

In [ ]:
selector_anova.fit(X_train_prep, y_train)

In [ ]:
columnas_anova = X_train_prep.columns[selector_anova.get_support()].tolist()
columnas_anova

In [ ]:
puntuaciones_anova = pd.Series(selector_anova.scores_, index=X_train_prep.columns)
puntuaciones_anova.sort_values(ascending=False).round(1).head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 8))
puntuaciones_anova.sort_values().plot.barh(ax=ax)
ax.set_title("ANOVA · Puntuación de cada columna en entrenamiento")
ax.set_xlabel("Puntuación F")
plt.tight_layout()
plt.show()

**Qué veo.** Las 6 que se quedan son `cp_asymptomatic` (233.5), `exang_no` (177.5), `exang_sí` (166.6), `cp_atypical angina` (131.7), `thalch` (128.4) y `oldpeak` (111.8). Pasa lo que advierte la clase: ANOVA no compara las columnas entre sí, así que se queda con `exang_no` y `exang_sí`, que casi dicen lo mismo (0.876 de correlación). `sex_Male` y `sex_Female` también tienen la misma puntuación (75.3).

---
## 4. Selección hacia adelante (forward)

Igual que en la clase: el selector empieza con la columna que da el mejor resultado en validación cruzada, luego prueba agregar cada una de las demás y acepta la mejor si sube al menos `1e-6`. El modelo que evalúa es el mismo SVC de la clase, con el escalado dentro del pipeline, y la validación cruzada usa 3 partes de entrenamiento. Mido con AUC.

In [ ]:
modelo_seleccion = Pipeline([
    ("scale", StandardScaler()),
    ("svc", SVC(kernel="rbf", C=1)),
])

In [ ]:
cv_seleccion = StratifiedKFold(
    n_splits=3, shuffle=True, random_state=RANDOM_STATE,
)

In [ ]:
selector_forward = SequentialFeatureSelector(
    modelo_seleccion,
    n_features_to_select="auto",
    tol=1e-6,
    direction="forward",
    scoring="roc_auc",
    cv=cv_seleccion,
)

In [ ]:
selector_forward.fit(X_train_prep, y_train)

In [ ]:
columnas_forward = X_train_prep.columns[selector_forward.get_support()].tolist()
print(len(columnas_forward), "columnas")
columnas_forward

Para entender el primer paso lo hago a mano: pruebo cada columna sola con el mismo modelo y la misma validación cruzada.

In [ ]:
auc_una_columna = {}
for columna in X_train_prep.columns:
    auc_una_columna[columna] = cross_val_score(
        modelo_seleccion, X_train_prep[[columna]], y_train, cv=cv_seleccion, scoring="roc_auc",
    ).mean()

pd.Series(auc_una_columna).sort_values(ascending=False).round(4).head()

In [ ]:
# El selector con una sola columna debe elegir la misma que salió arriba
primer_paso = SequentialFeatureSelector(
    modelo_seleccion, n_features_to_select=1, direction="forward", scoring="roc_auc", cv=cv_seleccion,
)
primer_paso.fit(X_train_prep, y_train)
X_train_prep.columns[primer_paso.get_support()].tolist()

**Qué veo.** Forward se queda con 15 columnas. El primer paso coincide con el cálculo a mano: sola, `cp_asymptomatic` es la que da mayor AUC (0.7464), y el selector con una sola columna también la elige.

Aunque `exang_no` es la segunda mejor sola (0.7199), forward no agrega ninguna columna de `exang`: con las columnas que ya tenía, en ningún paso fue la que más subía el AUC.

---
## 5. Selección hacia atrás (backward)

Parte de las 28 columnas y en cada paso quita la que deja el mejor AUC. Con `tol=0.0` sigue quitando mientras el AUC se mantenga o suba.

In [ ]:
selector_backward = SequentialFeatureSelector(
    modelo_seleccion,
    n_features_to_select="auto",
    tol=0.0,
    direction="backward",
    scoring="roc_auc",
    cv=cv_seleccion,
)

In [ ]:
selector_backward.fit(X_train_prep, y_train)

In [ ]:
columnas_backward = X_train_prep.columns[selector_backward.get_support()].tolist()
print(len(columnas_backward), "columnas")
columnas_backward

In [ ]:
print("En ambos:", sorted(set(columnas_forward) & set(columnas_backward)))
print("Solo forward:", sorted(set(columnas_forward) - set(columnas_backward)))
print("Solo backward:", sorted(set(columnas_backward) - set(columnas_forward)))

**Qué veo.** Backward se queda con 18 columnas. Las dos coinciden en 11, entre ellas las 5 numéricas, `cp_asymptomatic` y `sex_Female`. Forward tiene 4 que backward no y backward tiene 7 que forward no, así que llegan a subconjuntos distintos aunque usen el mismo modelo y la misma métrica. Ninguno vuelve a revisar lo que ya decidió.

---
## 6. Eliminación recursiva: RFE

RFE entrena el modelo, quita la columna con el coeficiente más chico (en valor absoluto) y vuelve a entrenar hasta dejar 6. Como en la clase, uso un SVC lineal, que sí tiene un coeficiente por columna, y estandarizo antes con un escalador ajustado solo con entrenamiento.

In [ ]:
escalador_seleccion = StandardScaler()
X_train_scaled = escalador_seleccion.fit_transform(X_train_prep)

In [ ]:
selector_rfe = RFE(
    estimator=SVC(kernel="linear", C=1),
    n_features_to_select=N_COLUMNAS,
    step=1,
)

In [ ]:
selector_rfe.fit(X_train_scaled, y_train)

In [ ]:
columnas_rfe = X_train_prep.columns[selector_rfe.get_support()].tolist()
columnas_rfe

In [ ]:
ranking_rfe = pd.Series(selector_rfe.ranking_, index=X_train_prep.columns)
ranking_rfe.sort_values().head(10)

**Qué veo.** RFE se queda con 6 columnas de 0 y 1 y ninguna numérica: `sex_Male`, `cp_atypical angina`, `cp_non-anginal`, `fbs_faltante`, `exang_no` y `slope_flat`. Las últimas que quitó antes de llegar a 6 fueron `age` (rango 2), `oldpeak` (3), `thal_reversable defect` (4) y `thalch` (5).

Me llama la atención `fbs_faltante`. En el proyecto vi que de los 90 pacientes sin `fbs`, 75 son de Suiza, así que esta columna puede estar marcando el hospital y no algo del paciente.

---
## 7. Selección con Lasso

Lasso es una regresión que puede dejar coeficientes en cero. Uso las columnas estandarizadas y `alpha=0.05`, como en la clase. Aquí el target ya es 0 o 1, así que basta una sola regresión (en Wine había una por clase).

In [ ]:
lasso = Lasso(alpha=0.05)
lasso.fit(X_train_scaled, y_train)

In [ ]:
pesos_lasso = pd.Series(lasso.coef_, index=X_train_prep.columns)
pesos_lasso[pesos_lasso != 0].round(3).sort_values()   # solo los que no quedaron en cero

In [ ]:
columnas_lasso = X_train_prep.columns[pesos_lasso != 0].tolist()
print(len(columnas_lasso), "columnas")
columnas_lasso

**Qué veo.** Con `alpha=0.05` quedan 12 columnas distintas de cero. El peso más grande es el de `cp_asymptomatic` (0.095), que sube la predicción de enfermedad; `exang_no` (-0.066) y `sex_Female` (-0.044) la bajan. `trestbps`, `chol` y todas las de `restecg` quedan en cero. También queda `fbs_faltante` (0.020), la misma pista del hospital que salió en RFE.

---
## 8. Extensión: búsqueda exhaustiva

Con 28 columnas habría 2**28 - 1 subconjuntos, más de 268 millones. Como en la clase, fijo antes 5 candidatas y pruebo sus 31 subconjuntos con el SVC fijo y validación cruzada sobre entrenamiento. Tomo como candidatas las 5 con mayor puntuación de ANOVA, que salen solo de entrenamiento.

In [ ]:
def subset_auc(columnas):
    puntajes = cross_val_score(
        modelo_seleccion, X_train_prep[columnas], y_train,
        cv=cv_seleccion, scoring="roc_auc",
    )
    return puntajes.mean()

In [ ]:
candidatas = puntuaciones_anova.sort_values(ascending=False).index[:5].tolist()
print(candidatas)
2 ** len(candidatas) - 1

In [ ]:
resultados_exhaustiva = []

for tamano in range(1, len(candidatas) + 1):
    for subconjunto in combinations(candidatas, tamano):
        columnas = list(subconjunto)
        resultados_exhaustiva.append({
            "columnas": columnas,
            "variables": len(columnas),
            "auc_cv": subset_auc(columnas),
        })

In [ ]:
tabla_exhaustiva = pd.DataFrame(resultados_exhaustiva)
tabla_exhaustiva = tabla_exhaustiva.sort_values(["auc_cv", "variables"], ascending=[False, True])
tabla_exhaustiva.head()

In [ ]:
columnas_exhaustiva = tabla_exhaustiva.iloc[0]["columnas"]
columnas_exhaustiva

**Qué veo.** Entre las 5 candidatas, el mejor subconjunto tiene 4 columnas: `cp_asymptomatic`, `exang_sí`, `cp_atypical angina` y `thalch`, con AUC de 0.8048. Quita `exang_no`, que repetía lo de `exang_sí`; con las 5 baja a 0.8017. Es un AUC más bajo que el de las otras técnicas. Hay que tomar en cuenta que usa solo 3 variables y que la búsqueda solo es exhaustiva dentro de esas 5 candidatas.

---
## 9. ¿Qué columnas eligió cada técnica?

Cada fila es una columna de 0 y 1 o numérica. Un 1 quiere decir que la técnica la conservó.

In [ ]:
conjuntos = {
    "Sin selección": X_train_prep.columns.tolist(),
    "Varianza cero": columnas_varianza,
    "Correlación": columnas_correlacion,
    "ANOVA": columnas_anova,
    "Hacia adelante": columnas_forward,
    "Hacia atrás": columnas_backward,
    "RFE": columnas_rfe,
    "Lasso": columnas_lasso,
    "Exhaustiva (5 candidatas)": columnas_exhaustiva,
}

In [ ]:
matriz_columnas = pd.DataFrame(index=X_train_prep.columns)
for tecnica, columnas in conjuntos.items():
    matriz_columnas[tecnica] = X_train_prep.columns.isin(columnas)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 11))
sns.heatmap(matriz_columnas.astype(int), annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax)
ax.set_title("Columnas seleccionadas · 1: conservada, 0: descartada")
ax.set_xlabel("Técnica de selección")
ax.set_ylabel("Columna")
plt.xticks(rotation=30, ha="right")
plt.tight_layout()
plt.show()

Una técnica puede quedarse con `cp_asymptomatic` y quitar las otras categorías de `cp`, pero para usar el modelo igual hay que preguntarle al paciente su tipo de dolor. Por eso cuento también cuántas de las 12 variables originales necesita cada técnica.

In [ ]:
for tecnica, columnas in conjuntos.items():
    variables = []
    for columna in columnas:
        variable = columna.split("__")[1].split("_")[0]   # cat__cp_asymptomatic -> cp
        if variable not in variables:
            variables.append(variable)
    faltan = [v for v in NUMERICAS + CATEGORICAS if v not in variables]
    print(f"{tecnica}: {len(columnas)} columnas, {len(variables)} variables | no usa: {faltan}")

**Qué veo.** Cada técnica que usa el target se queda con algo distinto. Lo que se repite:

- `cp_asymptomatic`, `cp_atypical angina` y `thalch` las conservan cinco de las seis técnicas que usan el target.
- `fbs_sí`, `restecg_lv hypertrophy`, `restecg_normal`, `slope_faltante` y `thal_fixed defect` no las conserva ninguna.
- `trestbps` y `chol` solo las conservan forward y backward.

Contando variables y no columnas, hacia atrás usa las 12 y hacia adelante 11 (no usa `exang`). Lasso usa 9 y deja fuera `trestbps`, `chol` y `restecg`. ANOVA, RFE y la exhaustiva usan entre 3 y 5.

---
## 10. Entrenar SVC, regresión logística y Random Forest

Como en la clase: las mismas 3 particiones para todos los modelos, tres valores de `C` para SVC y regresión logística y tres profundidades para Random Forest. Mido con AUC. Las columnas se eligieron con todo el entrenamiento, así que estos resultados pueden salir optimistas; la prueba queda para el final.

In [ ]:
model_cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
c_values = [0.1, 1, 10]

In [ ]:
svc = Pipeline([("escala", StandardScaler()), ("svc", SVC(kernel="rbf"))])
logistic = Pipeline([("escala", StandardScaler()), ("logistic", LogisticRegression(max_iter=2000, random_state=RANDOM_STATE))])
forest = RandomForestClassifier(n_estimators=100, random_state=RANDOM_STATE)

In [ ]:
candidatos = [
    ("SVC", svc, {"svc__C": c_values}),
    ("Regresión logística", logistic, {"logistic__C": c_values}),
    ("Random Forest", forest, {"max_depth": [3, 5, None]}),
]

In [ ]:
resultados = []
modelos_entrenados = {}
for tecnica, columnas in conjuntos.items():
    for nombre, modelo, grid in candidatos:
        busqueda = GridSearchCV(modelo, grid, cv=model_cv, scoring="roc_auc")
        busqueda.fit(X_train_prep[columnas], y_train)
        modelos_entrenados[(tecnica, nombre)] = busqueda.best_estimator_
        resultados.append({
            "técnica": tecnica, "columnas": len(columnas), "modelo": nombre,
            "auc_cv": busqueda.best_score_, "parámetros": busqueda.best_params_,
        })

### Comparación de resultados

In [ ]:
tabla_resultados = pd.DataFrame(resultados)
comparacion = tabla_resultados.pivot(index=["técnica", "columnas"], columns="modelo", values="auc_cv")
comparacion = comparacion.reindex(conjuntos.keys(), level="técnica")
comparacion[["SVC", "Regresión logística", "Random Forest"]].round(4)

**Qué veo.** El mayor AUC en validación cruzada es hacia atrás con SVC (0.8909), seguido de hacia adelante con SVC (0.8896) y de correlación y sin selección con SVC (0.8885 y 0.8884). Entre estos la diferencia es de menos de 0.003. En regresión logística el mejor es Lasso (0.8865, contra 0.8814 con las 28). ANOVA, RFE y la exhaustiva, que usan pocas variables, bajan a entre 0.81 y 0.87.

## Evaluación final

La regla de la clase: elijo el mayor AUC en validación cruzada, con los valores completos, y si hay empate prefiero el subconjunto con menos columnas. Después mido el modelo elegido en prueba.

In [ ]:
ordenados = tabla_resultados.sort_values(["auc_cv", "columnas"], ascending=[False, True], kind="stable")
mejor = ordenados.iloc[0]

In [ ]:
tecnica_ganadora = mejor["técnica"]
modelo_ganador = mejor["modelo"]
columnas_ganadoras = conjuntos[tecnica_ganadora]
print(tecnica_ganadora, "·", modelo_ganador, "·", mejor["parámetros"], "·", len(columnas_ganadoras), "columnas")

In [ ]:
modelo_final = modelos_entrenados[(tecnica_ganadora, modelo_ganador)]

In [ ]:
y_pred = modelo_final.predict(X_test_prep[columnas_ganadoras])

In [ ]:
print("Accuracy en prueba:", round(accuracy_score(y_test, y_pred), 4))
print("AUC en prueba:", round(roc_auc_score(y_test, modelo_final.decision_function(X_test_prep[columnas_ganadoras])), 4))

In [ ]:
print(classification_report(
    y_test, y_pred, labels=[0, 1], target_names=["sin enfermedad", "con enfermedad"],
    digits=3, zero_division=0,
))

In [ ]:
matriz = confusion_matrix(y_test, y_pred, labels=[0, 1])

In [ ]:
fig, ax = plt.subplots()
sns.heatmap(
    matriz, annot=True, fmt="d", cmap="Blues", cbar=False, ax=ax,
    xticklabels=["sin enfermedad", "con enfermedad"], yticklabels=["sin enfermedad", "con enfermedad"],
)
ax.set_title(f"Enfermedad cardiaca · {tecnica_ganadora} + {modelo_ganador}")
ax.set_xlabel("Clase predicha")
ax.set_ylabel("Clase real")
plt.tight_layout()
plt.show()

Solo para comparar, mido también el mismo modelo con las 28 columnas. Esto no cambia la elección.

In [ ]:
modelo_sin_seleccion = modelos_entrenados[("Sin selección", modelo_ganador)]
columnas_todas = conjuntos["Sin selección"]
y_pred_todas = modelo_sin_seleccion.predict(X_test_prep[columnas_todas])
print("Accuracy en prueba:", round(accuracy_score(y_test, y_pred_todas), 4))
print("AUC en prueba:", round(roc_auc_score(y_test, modelo_sin_seleccion.decision_function(X_test_prep[columnas_todas])), 4))
confusion_matrix(y_test, y_pred_todas, labels=[0, 1])

**Qué veo.** Con la regla de la clase gana hacia atrás con SVC (`C = 1`, 18 columnas). En prueba acierta 0.8424 (155 de 184) con AUC de 0.8975. Detecta 92 de los 102 enfermos (recall de 0.902): se le escapan 10 y marca como enfermos a 19 sanos.

El mismo SVC con las 28 columnas acierta menos (0.8261), pero tiene mayor AUC (0.9182) y se le escapan 8 enfermos, a cambio de 24 sanos mal marcados. Con 184 pacientes, 2 o 3 pacientes de diferencia no alcanzan para decir que uno es mejor. Como referencia, la regresión logística del proyecto, con estos mismos 184 pacientes de prueba, sacó 0.8533 de accuracy, 0.9077 de AUC y se le escaparon 9 enfermos, aunque se entrenó con menos pacientes (552).

---
## Conclusión

**¿Necesito todas las variables?** No todas. La regla de la clase eligió hacia atrás con SVC, que en validación cruzada queda apenas arriba de usar las 28 columnas (0.8909 contra 0.8884), pero sus 18 columnas siguen necesitando las 12 variables. Donde sí se ahorran mediciones es con Lasso: usa 12 columnas y 9 variables (deja fuera `trestbps`, `chol` y `restecg`), y con ellas la regresión logística saca 0.8865, más que con las 28 (0.8814). En prueba, el modelo elegido no le gana claramente al que usa todo.

Lo que más se repite entre técnicas es que el tipo de dolor de pecho (`cp`) y la frecuencia cardiaca máxima (`thalch`) sirven, y que `restecg` (electrocardiograma en reposo) casi nunca se conserva.

## Decisiones

| # | Decisión | Elección | Por qué |
|---|---|---|---|
| 1 | Datos y limpieza | Los del proyecto | Continuar el proyecto 1 |
| 2 | División | 80-20 como en la clase | La prueba son los mismos 184 pacientes del proyecto |
| 3 | Categóricas | `OneHotEncoder` con `faltante` | Las técnicas necesitan números sin faltantes |
| 4 | Métrica | AUC | Decirle enfermo a todos saca 0.8608 de F2 |
| 5 | Modelo final | Hacia atrás + SVC, `C = 1` | Mayor AUC en validación cruzada (0.8909) |

## Limitaciones

- Las columnas y la mediana se ajustaron con todo el entrenamiento, así que la validación cruzada de la sección 10 puede salir optimista.
- Todo sale de un solo reparto de 736 pacientes. Con otro reparto las técnicas podrían elegir otras columnas.
- `fbs_faltante` aparece en forward, RFE y Lasso, y puede estar marcando el hospital de Suiza en vez de algo del paciente.
- Seleccionar columnas de 0 y 1 puede dejar solo algunas categorías de una variable; para usar el modelo igual hay que medir la variable completa.